<a href="https://colab.research.google.com/github/peterbabulik/O-LLM/blob/main/O_LLM_ShakespeareDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ==============================================================================
# OCTONIONIC LARGE LANGUAGE MODEL (O-LLM)
# A Non-Associative, Fano-Algebraic Sequence Model for Constant-Memory Generation
# Compatible with PyTorch 2.0+ (Google Colab GPU / TPU / CPU)
# ==============================================================================

# %% [1] Install dependencies (if needed) & Import libraries
import os
import math
import urllib.request
import torch
import torch.nn as nn
import torch.nn.functional as F

# Device configuration (Auto-detect CUDA GPU on Colab)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[*] Running on device: {device}")
if torch.cuda.is_available():
    print(f"[*] GPU Name: {torch.cuda.get_device_name(0)}")

# %% [2] Algebraic Core: Vectorized Fano Multiplication Tensor for Octonions (O)
def build_fano_tensor() -> torch.Tensor:
    """
    Constructs the 8x8x8 structural multiplication tensor C_ijk for octonions,
    where: e_i * e_j = sum_k C_ijk * e_k.
    Index 0: e_0 = 1 (Scalar identity)
    Indices 1..7: e_1..e_7 (Imaginary units governed by the Fano plane)
    """
    C = torch.zeros((8, 8, 8), dtype=torch.float32)

    # 1. Real identity: e_0 * e_i = e_i, e_i * e_0 = e_i
    for i in range(8):
        C[0, i, i] = 1.0
        C[i, 0, i] = 1.0

    # 2. Imaginary squares: e_i^2 = -1 (for i >= 1)
    for i in range(1, 8):
        C[i, i, 0] = -1.0

    # 3. Oriented Fano plane triads (e_i * e_j = e_k)
    fano_triads = [
        (1, 2, 3), (1, 4, 5), (1, 7, 6),
        (2, 4, 6), (2, 5, 7), (3, 4, 7), (3, 6, 5)
    ]

    for i, j, k in fano_triads:
        # Cyclic permutations (+1)
        C[i, j, k] = 1.0;  C[j, k, i] = 1.0;  C[k, i, j] = 1.0
        # Anti-cyclic permutations (-1)
        C[j, i, k] = -1.0; C[k, j, i] = -1.0; C[i, k, j] = -1.0

    return C

class OctonionicAlgebra(nn.Module):
    """
    Batched non-associative octonionic operations via Einstein summation.
    """
    def __init__(self):
        super().__init__()
        self.register_buffer("C", build_fano_tensor())

    def multiply(self, A: torch.Tensor, B: torch.Tensor) -> torch.Tensor:
        # A, B: (..., 8) -> Out: (..., 8)
        return torch.einsum("...i,...j,ijk->...k", A, B, self.C)

    def associator(self, A: torch.Tensor, B: torch.Tensor, C: torch.Tensor) -> torch.Tensor:
        """
        Calculates the ternary Octonionic Associator:
        [A, B, C] = (A * B) * C - A * (B * C)
        """
        AB = self.multiply(A, B)
        AB_C = self.multiply(AB, C)

        BC = self.multiply(B, C)
        A_BC = self.multiply(A, BC)

        res = AB_C - A_BC
        norm = torch.linalg.norm(res, dim=-1, keepdim=True) + 1e-6
        return res / norm

# %% [3] Architecture: Octonionic Recurrent Attention Cell (O-Cell)
class OctonionicCell(nn.Module):
    """
    Recurrent sequence layer with O(1) memory complexity.
    Partitioned into C = d_model // 8 independent 8D octonionic channels.
    """
    def __init__(self, d_model: int):
        super().__init__()
        assert d_model % 8 == 0, "d_model must be a multiple of 8."
        self.d_model = d_model
        self.num_channels = d_model // 8
        self.algebra = OctonionicAlgebra()

        # Learnable chiral gate bias initialized with alternating parity
        gate_init = torch.tensor([1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0] * self.num_channels)
        gate_init = gate_init.view(self.num_channels, 8)
        gate_init = gate_init / torch.linalg.norm(gate_init, dim=-1, keepdim=True)
        self.e_gate = nn.Parameter(gate_init)

        # Asymmetric skip bypass per channel
        self.W_bypass = nn.Parameter(torch.randn(self.num_channels, 8, 8) * 0.05)

        # Learnable gating interpolation parameters
        self.raw_alpha = nn.Parameter(torch.zeros(self.num_channels, 1))
        self.raw_beta = nn.Parameter(torch.zeros(self.num_channels, 1))

    def step(self, h_prev: torch.Tensor, x_t: torch.Tensor) -> torch.Tensor:
        alpha = torch.sigmoid(self.raw_alpha)
        beta = torch.sigmoid(self.raw_beta)

        # 1. Non-associative associator innovation across Fano plane
        gate = self.e_gate.unsqueeze(0) # (1, C, 8)
        delta_h = self.algebra.associator(h_prev, x_t, gate)

        # 2. Linear bypass
        bypass = torch.einsum("bci,cij->bcj", x_t, self.W_bypass)

        # 3. State synthesis & Unitary spherical normalization
        h_next = (1.0 - alpha) * h_prev + alpha * delta_h + beta * bypass
        norm = torch.linalg.norm(h_next, dim=-1, keepdim=True) + 1e-6
        return h_next / norm

    def forward(self, x: torch.Tensor, h_init: torch.Tensor = None) -> tuple[torch.Tensor, torch.Tensor]:
        B, T, D = x.shape
        x_channels = x.view(B, T, self.num_channels, 8)

        if h_init is None:
            # Initialize with e_0 = 1 scalar identity
            h_curr = torch.zeros((B, self.num_channels, 8), device=x.device, dtype=x.dtype)
            h_curr[..., 0] = 1.0
        else:
            h_curr = h_init

        h_states = []
        for t in range(T):
            x_t = x_channels[:, t, :, :]
            h_curr = self.step(h_curr, x_t)
            h_states.append(h_curr)

        out = torch.stack(h_states, dim=1).view(B, T, D)
        return out, h_curr

# %% [4] Full Causal Octonionic Language Model (O-LLM)
class OctonionicLLM(nn.Module):
    def __init__(self, vocab_size: int, d_model: int = 64, num_layers: int = 2):
        super().__init__()
        assert d_model % 8 == 0, "d_model must be a multiple of 8."
        self.vocab_size = vocab_size
        self.d_model = d_model
        self.num_layers = num_layers

        self.tok_embeddings = nn.Embedding(vocab_size, d_model)
        self.layers = nn.ModuleList([OctonionicCell(d_model) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)

        # Weight tying
        self.head.weight = self.tok_embeddings.weight
        nn.init.normal_(self.tok_embeddings.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, layer_states: list = None):
        x = self.tok_embeddings(idx)
        new_states = []
        for l_idx, layer in enumerate(self.layers):
            h_prev = layer_states[l_idx] if layer_states is not None else None
            x, h_next = layer(x, h_init=h_prev)
            new_states.append(h_next)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits, new_states

    @torch.no_grad()
    def generate(self, prompt_tokens: torch.Tensor, max_new_tokens: int = 64, temperature: float = 0.7):
        self.eval()
        B, T = prompt_tokens.shape
        logits, states = self.forward(prompt_tokens)

        next_token_logits = logits[:, -1, :] / max(temperature, 1e-5)
        probs = F.softmax(next_token_logits, dim=-1)
        next_tok = torch.multinomial(probs, num_samples=1)
        generated = [next_tok]

        current_token = next_tok
        for _ in range(max_new_tokens - 1):
            x = self.tok_embeddings(current_token)
            x_channels = x.view(B, 1, self.d_model // 8, 8)

            new_states = []
            for l_idx, layer in enumerate(self.layers):
                h_prev = states[l_idx]
                h_next = layer.step(h_prev, x_channels[:, 0, :, :])
                new_states.append(h_next)
                x_channels = h_next.unsqueeze(1)

            states = new_states
            h_final = states[-1].view(B, 1, self.d_model)
            logits = self.head(self.ln_f(h_final))[:, 0, :] / max(temperature, 1e-5)
            probs = F.softmax(logits, dim=-1)
            current_token = torch.multinomial(probs, num_samples=1)
            generated.append(current_token)

        return torch.cat([prompt_tokens] + generated, dim=1)

# %% [5] Dataset Preparation: Tiny Shakespeare
print("\n[*] Downloading and tokenizing Tiny Shakespeare dataset...")
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text_file = "tinyshakespeare.txt"
if not os.path.exists(text_file):
    urllib.request.urlretrieve(url, text_file)

with open(text_file, "r", encoding="utf-8") as f:
    raw_data = f.read()

# Build character vocabulary
chars = sorted(list(set(raw_data)))
vocab_size = len(chars)
char_to_id = {ch: i for i, ch in enumerate(chars)}
id_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_id[c] for c in s]
decode = lambda l: ''.join([id_to_char[i] for i in l])

data_tensor = torch.tensor(encode(raw_data), dtype=torch.long)
n_train = int(0.9 * len(data_tensor))
train_data = data_tensor[:n_train]
val_data = data_tensor[n_train:]

print(f"[*] Total Characters: {len(data_tensor):,}, Vocabulary Size: {vocab_size}")

def get_batch(split="train", batch_size=32, block_size=64):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix]).to(device)
    y = torch.stack([d[i+1:i+block_size+1] for i in ix]).to(device)
    return x, y

# %% [6] Training Loop on Colab
d_model = 64     # 8 octonionic channels (8 x 8)
num_layers = 2
batch_size = 32
block_size = 64
max_steps = 1000
eval_interval = 200
learning_rate = 3e-3

model = OctonionicLLM(vocab_size=vocab_size, d_model=d_model, num_layers=num_layers).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-2)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"[*] O-LLM initialized with {total_params:,} parameters.")
print("[*] Starting training...")

model.train()
for step in range(1, max_steps + 1):
    xb, yb = get_batch("train", batch_size, block_size)
    logits, _ = model(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    if step % eval_interval == 0 or step == max_steps:
        model.eval()
        with torch.no_grad():
            xv, yv = get_batch("val", batch_size, block_size)
            v_logits, _ = model(xv)
            v_loss = F.cross_entropy(v_logits.view(-1, vocab_size), yv.view(-1))
            ppl = math.exp(v_loss.item()) if v_loss.item() < 20 else float('inf')
        print(f"Step {step:4d}/{max_steps} | Train Loss: {loss.item():.4f} | Val Loss: {v_loss.item():.4f} | Val Perplexity: {ppl:.2f}")
        model.train()

print("[*] Training completed successfully!")

# %% [7] Autoregressive Generation Test (O(1) Constant Memory)
print("\n" + "="*70)
print("AUTOREGRESSIVE GENERATION DEMO (CONSTANT 32-BYTE MEMORY CACHE)")
print("="*70)
prompt_text = "KING: Shall I "
prompt_idx = torch.tensor([encode(prompt_text)], dtype=torch.long).to(device)

generated_idx = model.generate(prompt_idx, max_new_tokens=150, temperature=0.75)
output_text = decode(generated_idx[0].tolist())
print(output_text)
print("="*70)

# %% [8] Save Model Checkpoint
torch.save({
    'model_state_dict': model.state_dict(),
    'vocab_size': vocab_size,
    'char_to_id': char_to_id,
    'id_to_char': id_to_char,
    'd_model': d_model,
    'num_layers': num_layers
}, "octonionic_llm_checkpoint.pt")
print("[*] Checkpoint saved as 'octonionic_llm_checkpoint.pt'")

[*] Running on device: cuda
[*] GPU Name: Tesla T4

[*] Downloading and tokenizing Tiny Shakespeare dataset...
[*] Total Characters: 1,115,394, Vocabulary Size: 65
[*] O-LLM initialized with 5,472 parameters.
[*] Starting training...
Step  200/1000 | Train Loss: 3.2857 | Val Loss: 3.2991 | Val Perplexity: 27.09
Step  400/1000 | Train Loss: 2.9159 | Val Loss: 2.9538 | Val Perplexity: 19.18
Step  600/1000 | Train Loss: 2.5781 | Val Loss: 2.5692 | Val Perplexity: 13.06
Step  800/1000 | Train Loss: 2.4410 | Val Loss: 2.4532 | Val Perplexity: 11.63
Step 1000/1000 | Train Loss: 2.3525 | Val Loss: 2.3887 | Val Perplexity: 10.90
[*] Training completed successfully!

AUTOREGRESSIVE GENERATION DEMO (CONSTANT 32-BYTE MEMORY CACHE)
KING: Shall I now amsensst tilce have ill thi tet the prethe ralesou, paset hirte bathin ton tale he bind peats ares mon ar sear maveres that of o yore dor ly ably 
[*] Checkpoint saved as 'octonionic_llm_checkpoint.pt'
